# Mixed Variables & Handling Date-Time — Feature Engineering Theory

Two separate feature-engineering topics: how to handle a column that mixes numbers and categories, and how to turn a raw timestamp into features a model can actually use.

```
                Feature Engineering
                        |
        ┌───────────────┴───────────────┐
        |                                |
  Mixed Variables                  Date & Time
        |                                |
 ├─ Numbers-only rows            ├─ Extract components
 └─ Strings-only rows            ├─ Elapsed time
 (combined num+cat values         └─ Cyclical encoding
  like "A16", "C23")
```

---

## 1. Mixed Variables

**What it is:** A column that contains **both numbers and categories** — either as separate values in different rows, or combined together inside the same value (e.g. Titanic's `Cabin` column: `A16`, `C23`, `G6`). Models can't use such a column directly, so it's split into a clean **numerical** part and a clean **categorical** part.

**Two common patterns:**
| Pattern | Example | Meaning |
|---|---|---|
| Number **OR** string per row | `1`, `2`, `D`, `3`, `G` | Some rows are purely numeric, others are purely categorical |
| Number **AND** string combined in one value | `A16`, `C23`, `G6` | Each value itself mixes a letter (deck/category) and a number (position) |

```
BEFORE (raw mixed column)          AFTER (split into two clean columns)
--------------------------          -----------------------------------
cabin                                cabin_num   cabin_cat
-----                                ---------   ---------
A16                                     16           A
C23                                     23           C
D                                      NaN           D
1                                        1           1
2                                        2           2
B45                                     45           B
G6                                       6           G
3                                        3           3
E12                                     12           E
A26                                     26           A
NaN                                    NaN          NaN
C85                                     85           C
```

**Note:** After splitting, `cabin_num` goes through normal **numerical preprocessing** (scaling, binning, imputation) and `cabin_cat` goes through normal **categorical preprocessing** (encoding). The split itself is the whole trick — everything downstream is standard.

**Key points:**
- Detect mixed variables by checking if a column's dtype is `object` but contains a **mix of digit-only and non-digit strings**
- Splitting = **extract digits** → numeric column, **extract letter/prefix** → categorical column
- No information is lost — the column is only reshaped, not reduced

---

## 2. Handling Date and Time

**What it is:** A raw `datetime` column (e.g. `2024-07-04 21:45:00`) is **not directly usable** by most ML models — it needs to be broken into meaningful numeric/categorical components first. `pandas`'s `.dt` accessor does this extraction.

**Common extracted features:**
| Feature | Use case |
|---|---|
| Year / Month / Day | Trend & seasonality |
| Day of week (Mon=0…Sun=6) | Weekday vs weekend patterns |
| Hour / Minute | Time-of-day patterns |
| Is weekend / month-start / month-end | Business-rule flags |
| Quarter | Reporting periods |
| Elapsed time (gap from a reference date) | Recency — "how long ago" |

```
BEFORE (raw timestamp)          AFTER (extracted components)
------------------------------  ------------------------------------------------------
timestamp                       year month day dayofweek hour is_weekend quarter
2024-01-15 08:30:00             2024   1   15      0        8      0        1
2024-03-22 14:05:00             2024   3   22      4       14      0        1
2024-07-04 21:45:00             2024   7    4      3       21      0        3
2024-11-29 03:10:00             2024  11   29      4        3      0        4
2024-12-25 12:00:00             2024  12   25      2       12      0        4
```

**Elapsed time (recency feature)** — instead of (or in addition to) the components above, measure the **gap from a reference date**:
```
timestamp                   days_since
2024-01-15 08:30:00               351
2024-03-22 14:05:00               284
2024-07-04 21:45:00               180
2024-11-29 03:10:00                32
2024-12-25 12:00:00                 6
```

### ⚠️ Cyclical Encoding — the key gotcha
Raw numbers like `hour` (0–23) or `month` (1–12) are **cyclic**, not linear — hour `23` and hour `0` are actually right next to each other, but as plain integers they look maximally far apart. Feeding them in raw makes the model think Dec (12) and Jan (1) are unrelated. The fix is to encode with **sine and cosine**:

```
hour_sin = sin(2π × hour  / 24)      month_sin = sin(2π × month / 12)
hour_cos = cos(2π × hour  / 24)      month_cos = cos(2π × month / 12)
```

```
hour  hour_sin  hour_cos    month  month_sin  month_cos
  8     0.866    -0.500        1      0.500      0.866
 14    -0.500    -0.866        3      1.000      0.000
 21    -0.707     0.707        7     -0.500     -0.866
  3     0.707     0.707       11     -0.500      0.866
 12     0.000    -1.000       12     -0.000      1.000
```
Notice hour `21` and hour `3` (only 6 hours apart across midnight) land at very different raw values, but their `(sin, cos)` pair correctly places them on opposite sides of the circle — this is exactly what preserves "closeness" near the wrap-around point (23→0, Dec→Jan).

**Key properties:**
- Always extract components **before** dropping the original datetime column
- Use **sin/cos pairs** for anything cyclical: hour, day-of-week, month, day-of-year
- **Elapsed-time / recency** features (days since an event) are often more predictive than the raw date itself
- Watch out for **timezones** — normalize to one timezone before extracting components

---

## Quick comparison

| Aspect | Mixed Variables (split) | Date & Time (extraction) |
|---|---|---|
| Problem solved | Column mixes numbers & categories | Raw timestamp isn't model-usable |
| Technique | Regex/string split into 2 columns | `.dt` accessor + sin/cos encoding |
| Data loss? | None — just reshapes the column | None — adds derived columns |
| Key gotcha | Deciding what counts as the "category" part | Cyclical features need sin/cos, not raw integers |

In [64]:
import numpy as np
import pandas as pd
from fontTools.afmLib import error

pd.set_option('display.max_columns', None)

In [65]:
df = pd.read_csv(r'/home/mukul/Main/Basic_DataScience-4ML/assets/data/mixed.csv')

In [66]:
df.sample(10)

,ticket_id,date,mixed_variable,source,product_id,orders,msg
1509,T1487,2014-02-15 01:12:00,579,messages,NaN,NaN,ласковая кошечка ищет своего ночного котика
604,S940,2019-12-05 02:46:35,Z,orders,5708.0,1.0,NaN
565,Q9895,2019-04-29 01:22:09,G,orders,888.0,5.0,NaN
1720,S3727,2014-08-14 00:32:00,L,messages,NaN,NaN,Парень для алени 0962628965
1628,T6879,2013-10-12 00:39:00,Y,messages,NaN,NaN,нежно поцелую девушк ножк 0964241588 ЗАДЕРЖИ
1139,R8348,2012-10-16 01:51:00,910,messages,NaN,NaN,парень 25 из Днепра ищет девчонку для встреч 0...
687,D1571,2019-10-13 04:30:33,Z,orders,2242.0,4.0,NaN
1261,N4158,2016-02-25 22:36:00,J,messages,NaN,NaN,П.23. Познакомлюсь с девушкой для с/о. 0932423381
1096,C6481,2012-12-26 00:32:00,R,messages,NaN,NaN,ПАР ПРИГЛАШУ ИЛИ ПРИЕДУ К ДЕВ ДЛЯ ВСТРЕЧИ. СЕЙ...
1634,C5319,2015-03-18 23:46:00,368,messages,NaN,NaN,ОДИНОКИЙ П 30л ИЩЕТ ОДИНОКУЮ ДЕВ.С/О ДНЕПР т.0...


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ticket_id       2000 non-null   object 
 1   date            2000 non-null   object 
 2   mixed_variable  2000 non-null   object 
 3   source          2000 non-null   object 
 4   product_id      1000 non-null   float64
 5   orders          1000 non-null   float64
 6   msg             1000 non-null   object 
dtypes: float64(2), object(5)
memory usage: 109.5+ KB


**Extract Ticket**

here we extract num and char and each row having both num and row so index number will preserve and also there is no null row after extracting because each row have num and char

In [68]:
df['ticket_id_num'] = df['ticket_id'].str.extract(r"(\d+)").astype(np.int64)
df['ticket_id_str'] = df['ticket_id'].str.extract(r"([A-Z]+)")

In [69]:
df['ticket_id_num'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2000 entries, 0 to 1999
Series name: ticket_id_num
Non-Null Count  Dtype
--------------  -----
2000 non-null   int64
dtypes: int64(1)
memory usage: 15.8 KB


In [70]:
df['ticket_id_str'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2000 entries, 0 to 1999
Series name: ticket_id_str
Non-Null Count  Dtype 
--------------  ----- 
2000 non-null   object
dtypes: object(1)
memory usage: 15.8+ KB


Extract mixed variable

here each row have only one char or num so when we extract only string the row having num will be marked as null same for num ( if we dont take null here then it decrease row number )

In [71]:
df['mixed_variable_num'] = pd.to_numeric(df['mixed_variable'], errors='coerce')
df['mixed_variable_str'] = np.where(df['mixed_variable_num'].isnull(),df['mixed_variable'],np.nan)

In [72]:
df['mixed_variable_num'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2000 entries, 0 to 1999
Series name: mixed_variable_num
Non-Null Count  Dtype  
--------------  -----  
1011 non-null   float64
dtypes: float64(1)
memory usage: 15.8 KB


In [73]:
df['mixed_variable_str'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2000 entries, 0 to 1999
Series name: mixed_variable_str
Non-Null Count  Dtype 
--------------  ----- 
989 non-null    object
dtypes: object(1)
memory usage: 15.8+ KB


In [74]:
df.sample(10)

,ticket_id,date,mixed_variable,source,product_id,orders,msg,ticket_id_num,ticket_id_str,mixed_variable_num,mixed_variable_str
300,U7170,2019-05-31 07:24:51,C,orders,2981.0,1004.0,NaN,7170,U,NaN,C
472,V9108,2019-11-05 01:28:47,162,orders,7464.0,3.0,NaN,9108,V,162.0,NaN
1583,Q2630,2013-05-01 00:10:00,G,messages,NaN,NaN,ПАРЕНЬ ПОЗНАКОМИТСЯ С ДЕВУШКОЙ ДЛЯ С\\О ЗВОНИТ...,2630,Q,NaN,G
952,B397,2018-08-18 07:37:13,657,orders,6492.0,1.0,NaN,397,B,657.0,NaN
408,P957,2019-05-31 08:55:33,K,orders,1341.0,8.0,NaN,957,P,NaN,K
910,Z2054,2019-04-25 15:07:38,F,orders,2504.0,2.0,NaN,2054,Z,NaN,F
24,U8365,2019-07-24 22:59:55,659,orders,2303.0,113.0,NaN,8365,U,659.0,NaN
1598,Z3033,2012-09-26 01:01:00,N,messages,NaN,NaN,П. 20 Л. ПАЗНАКОМЕЦА С ДЕВУШКОЙ ДЛЯ С/О С ДНЕП...,3033,Z,NaN,N
565,Q9895,2019-04-29 01:22:09,G,orders,888.0,5.0,NaN,9895,Q,NaN,G
756,J1480,2018-08-30 21:56:02,709,orders,7011.0,6.0,NaN,1480,J,709.0,NaN


Handling Date-Time

In [75]:
df['date'] = df['date'].astype('datetime64[s]')

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype        
---  ------              --------------  -----        
 0   ticket_id           2000 non-null   object       
 1   date                2000 non-null   datetime64[s]
 2   mixed_variable      2000 non-null   object       
 3   source              2000 non-null   object       
 4   product_id          1000 non-null   float64      
 5   orders              1000 non-null   float64      
 6   msg                 1000 non-null   object       
 7   ticket_id_num       2000 non-null   int64        
 8   ticket_id_str       2000 non-null   object       
 9   mixed_variable_num  1011 non-null   float64      
 10  mixed_variable_str  989 non-null    object       
dtypes: datetime64[s](1), float64(3), int64(1), object(6)
memory usage: 172.0+ KB


In [77]:
df

,ticket_id,date,mixed_variable,source,product_id,orders,msg,ticket_id_num,ticket_id_str,mixed_variable_num,mixed_variable_str
0,D5036,2019-12-10 02:52:09,459,orders,5628.0,3.0,NaN,5036,D,459.0,NaN
1,O6330,2018-08-15 19:52:18,704,orders,3646.0,157.0,NaN,6330,O,704.0,NaN
2,U3554,2018-10-23 03:59:23,A,orders,1859.0,1.0,NaN,3554,U,NaN,A
3,S6556,2019-08-17 12:13:53,L,orders,7292.0,1.0,NaN,6556,S,NaN,L
4,D9401,2019-01-06 16:39:20,M,orders,4344.0,3.0,NaN,9401,D,NaN,M
...,...,...,...,...,...,...,...,...,...,...,...
1995,H9523,2012-03-16 00:50:00,K,messages,NaN,NaN,ПАРЕНЬ СДЕЛАЕТ МАССАЖ ЖЕНЩИНАМ -066-877-32-44,9523,H,NaN,K
1996,Q6037,2014-01-23 23:14:00,E,messages,NaN,NaN,сельский п 23 ищу девушку для отношений,6037,Q,NaN,E
1997,I6468,2012-10-15 23:37:00,D,messages,NaN,NaN,Д+Д ДЛЯ серьезных отношений. Мой номер 093-156...,6468,I,NaN,D
1998,K2781,2012-06-21 23:34:00,254,messages,NaN,NaN,7 ДНЕПР М.34 ПОЗ.С Д/Ж ДЛЯ ВСТРЕЧ.Т.098 809 15 14,2781,K,254.0,NaN


In [1]:
df['date_second']= df['date'].dt.second
df['date_minute']= df['date'].dt.minute
df['date_hour']= df['date'].dt.hour
df['date_day']=df['date'].dt.day
df['date_week']=df['date'].dt.dayofweek

NameError: name 'df' is not defined

In [89]:
df

,ticket_id,date,mixed_variable,source,product_id,orders,msg,ticket_id_num,ticket_id_str,mixed_variable_num,mixed_variable_str,date_second,date_day,date_week,date_minute,date_hour
0,D5036,2019-12-10 02:52:09,459,orders,5628.0,3.0,NaN,5036,D,459.0,NaN,9,10,1,52,2
1,O6330,2018-08-15 19:52:18,704,orders,3646.0,157.0,NaN,6330,O,704.0,NaN,18,15,2,52,19
2,U3554,2018-10-23 03:59:23,A,orders,1859.0,1.0,NaN,3554,U,NaN,A,23,23,1,59,3
3,S6556,2019-08-17 12:13:53,L,orders,7292.0,1.0,NaN,6556,S,NaN,L,53,17,5,13,12
4,D9401,2019-01-06 16:39:20,M,orders,4344.0,3.0,NaN,9401,D,NaN,M,20,6,6,39,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,H9523,2012-03-16 00:50:00,K,messages,NaN,NaN,ПАРЕНЬ СДЕЛАЕТ МАССАЖ ЖЕНЩИНАМ -066-877-32-44,9523,H,NaN,K,0,16,4,50,0
1996,Q6037,2014-01-23 23:14:00,E,messages,NaN,NaN,сельский п 23 ищу девушку для отношений,6037,Q,NaN,E,0,23,3,14,23
1997,I6468,2012-10-15 23:37:00,D,messages,NaN,NaN,Д+Д ДЛЯ серьезных отношений. Мой номер 093-156...,6468,I,NaN,D,0,15,0,37,23
1998,K2781,2012-06-21 23:34:00,254,messages,NaN,NaN,7 ДНЕПР М.34 ПОЗ.С Д/Ж ДЛЯ ВСТРЕЧ.Т.098 809 15 14,2781,K,254.0,NaN,0,21,3,34,23
